<a href="https://colab.research.google.com/github/MBee059/it-agent-/blob/main/ITAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install required packages
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps xformers trl peft accelerate bitsandbytes
!pip install -q sentence-transformers chromadb langchain-text-splitters

# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216.9 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 118.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 36.5 MB/s eta 0:00:

In [ ]:
import torch
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load fine-tuned adapter from Google Drive in 4-bit mode
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

# 2. Set up ChromaDB Vector DB with IT Knowledge Base
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)

documents = [
    {
        "id": "doc_001",
        "text": "Nginx 502 Bad Gateway Error Resolution:\nA 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
    }
]

chunked_docs, chunk_ids, metadatas = [], [], []
for doc in documents:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append(chunk)
        chunk_ids.append(f"{doc['id']}_c{i}")
        metadatas.append({"source": doc["id"]})

# Use CPU for embeddings to keep GPU entirely free for generation
embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(chunked_docs).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="it_knowledge_base")
collection.add(documents=chunked_docs, embeddings=embeddings, metadatas=metadatas, ids=chunk_ids)

# 3. Retrieve Context for Query
user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_embedding = embedding_model.encode([user_query]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=1)
retrieved_context = results["documents"][0][0]

# 4. Construct Grounded Prompt & Generate
rag_prompt = f"""<|im_start|>system
You are an expert IT support agent. Format your response strictly using standard IT resolution headers:
## Symptom
## Likely Cause
## Diagnostic Steps
## Resolution Steps
## Verification

CRITICAL INSTRUCTION: Use ONLY the provided context below to extract exact commands, file paths, and configuration steps for the resolution.

Context:
{retrieved_context}<|im_end|>
<|im_start|>user
{user_query}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([rag_prompt], return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=384, temperature=0.2)

response = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant\n")[-1]

print("=== RETRIEVED CONTEXT ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED CONTEXT ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.

========================= AGENT RESPONSE =========================
## Likely Cause
Misconfiguration, software bug, or missing environment settings.

## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. Nginx is returning 502 Bad Gateway after restarting PHP-FPM. - Stack Overflow
2. I have a VPS running Ubuntu 16.04 LTS, nginx 1.10.3, PHP 7.0.19 and MySQL 5.7.18. I am trying to run a Laravel app on it. I have followed the instructions given here: https://www

In [ ]:
import torch
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# SECTION 1: Load Model (Keep As Is)
# ==========================================
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "cuda",
)
FastLanguageModel.for_inference(model)

# ==========================================
# SECTION 2: Vector DB Setup (Keep As Is)
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)

documents = [
    {
        "id": "doc_001",
        "text": (
            "Nginx 502 Bad Gateway Error Resolution:\n"
            "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). "
            "Check if the php-fpm service is running using `systemctl status php-fpm`. "
            "Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
        )
    }
]

chunked_docs, chunk_ids, metadatas = [], [], []
for doc in documents:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append(chunk)
        chunk_ids.append(f"{doc['id']}_c{i}")
        metadatas.append({"source": doc["id"]})

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(chunked_docs).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="it_knowledge_base")
collection.add(documents=chunked_docs, embeddings=embeddings, metadatas=metadatas, ids=chunk_ids)

# ==========================================
# SECTION 3: Context Retrieval (Keep As Is)
# ==========================================
user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_embedding = embedding_model.encode([user_query]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=1)
retrieved_context = results["documents"][0][0]

# ==========================================
# SECTION 4: Properly Formatted Grounded Prompt
# ==========================================
messages = [
    {
        "role": "system",
        "content": (
            "You are an expert IT support agent. Format your response strictly using standard IT resolution headers:\n"
            "## Symptom\n## Likely Cause\n## Diagnostic Steps\n## Resolution Steps\n## Verification\n\n"
            "CRITICAL INSTRUCTION: Use ONLY the provided context below to extract exact commands, file paths, and configuration steps for the resolution.\n\n"
            f"Context:\n{retrieved_context}"
        )
    },
    {
        "role": "user",
        "content": user_query
    }
]

# Apply official Qwen tokenization
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# ==========================================
# SECTION 5: Greedy Decoding (No Hallucinations)
# ==========================================
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=384,
        do_sample=False,        # Greedy search prevents random web completion
        repetition_penalty=1.1
    )

# Slice input tokens to output only the model's new answer
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== RETRIEVED CONTEXT ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=384) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED CONTEXT ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.

========================= AGENT RESPONSE =========================
## Likely Cause
Misconfiguration, software bug, or missing environment settings.

## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. Nginx is returning 502 Bad Gateway after restarting PHP-FPM. - Stack Overflow
2. I have a VPS running Ubuntu 16.04 LTS and nginx 1.10.3. I am trying to run a Laravel app on it. The app runs fine until I restart the PHP-FPM service. After that, whenever I try 

In [ ]:
import torch
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ==========================================
# SECTION 1: Load Model (Keep As Is)
# ==========================================
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "cuda",
)
FastLanguageModel.for_inference(model)

# ==========================================
# SECTION 2: Vector DB Setup (Keep As Is)
# ==========================================
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)

documents = [
    {
        "id": "doc_001",
        "text": (
            "Nginx 502 Bad Gateway Error Resolution:\n"
            "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). "
            "Check if the php-fpm service is running using `systemctl status php-fpm`. "
            "Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
        )
    }
]

chunked_docs, chunk_ids, metadatas = [], [], []
for doc in documents:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append(chunk)
        chunk_ids.append(f"{doc['id']}_c{i}")
        metadatas.append({"source": doc["id"]})

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(chunked_docs).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="it_knowledge_base")
collection.add(documents=chunked_docs, embeddings=embeddings, metadatas=metadatas, ids=chunk_ids)

# ==========================================
# SECTION 3: Context Retrieval (Keep As Is)
# ==========================================
user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_embedding = embedding_model.encode([user_query]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=1)
retrieved_context = results["documents"][0][0]

# ==========================================
# SECTION 4 & 5: Few-Shot Grounded Extraction
# ==========================================

# 1. Construct prompt with explicit few-shot instruction
prompt = f"""<|im_start|>system
You are a factual IT extractor. Extract ONLY the diagnostic commands and file paths directly mentioned in the Context. Never add external links, curl commands, or general web tutorials.<|im_end|>
<|im_start|>user
Context:
Apache Error: Check status using `systemctl status apache2` and inspect `/etc/apache2/ports.conf`.

Query: How to fix Apache issue?<|im_end|>
<|im_start|>assistant
## Diagnostic Steps
1. Check service status: `systemctl status apache2`

## Resolution Steps
1. Verify configuration in `/etc/apache2/ports.conf`<|im_end|>
<|im_start|>user
Context:
{retrieved_context}

Query: {user_query}<|im_end|>
<|im_start|>assistant
"""

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 2. Enforce strict stopping and low generation length
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,              # Greedy search
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== RETRIEVED CONTEXT ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED CONTEXT ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.

========================= AGENT RESPONSE =========================
## Likely Cause
* The most common cause of this error code is an incorrect configuration setting.

## Diagnostic Steps
1. Review the system logs for more information about why your request failed.
2. If you're still having trouble resolving this problem despite following these instructions please submit a support ticket so our engineers can assist you further.

## Resolution Steps
1. nginx -v
2. nginx version: nginx/1.4.6 (Ubuntu)
3. ps aux | grep fpm
4. root      7942  0.0  0.0   4240    64 ?        Ss   Ma

In [ ]:
import torch
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Load fine-tuned adapter explicitly mapped to GPU
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "cuda",
)
FastLanguageModel.for_inference(model)

# 2. Vector DB Setup
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)

documents = [
    {
        "id": "doc_001",
        "text": (
            "Nginx 502 Bad Gateway Error Resolution:\n"
            "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). "
            "Check if the php-fpm service is running using `systemctl status php-fpm`. "
            "Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
        )
    }
]

chunked_docs, chunk_ids, metadatas = [], [], []
for doc in documents:
    chunks = text_splitter.split_text(doc["text"])
    for i, chunk in enumerate(chunks):
        chunked_docs.append(chunk)
        chunk_ids.append(f"{doc['id']}_c{i}")
        metadatas.append({"source": doc["id"]})

embedding_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = embedding_model.encode(chunked_docs).tolist()

chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection(name="it_knowledge_base")
collection.add(documents=chunked_docs, embeddings=embeddings, metadatas=metadatas, ids=chunk_ids)

# 3. Retrieve Context
user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_embedding = embedding_model.encode([user_query]).tolist()
results = collection.query(query_embeddings=query_embedding, n_results=1)
retrieved_context = results["documents"][0][0]

# 4. Strict Grounding via Official Chat Template
messages = [
    {
        "role": "system",
        "content": (
            "You are a strict RAG extraction agent. You must answer the user query using ONLY the provided documentation context.\n"
            "Extract exact commands and file paths directly from the text. Do NOT rely on memory or output generic server configs.\n\n"
            "Required Output Structure:\n"
            "## Diagnostic Steps\n"
            "## Resolution Steps"
        )
    },
    {
        "role": "user",
        "content": f"Documentation Context:\n{retrieved_context}\n\nUser Question: {user_query}"
    }
]

# Apply chat template tokenization
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 5. Greedy Search Generation (do_sample=False)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,              # Eliminates random sampling/hallucination
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== RETRIEVED CONTEXT ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)

==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED CONTEXT ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.

========================= AGENT RESPONSE =========================
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js).
2. Check if the php-fpm service is running using systemctl status php-fpm .
3. Verify that the unix socket path in /etc/nginx/sites-available/default matches the listen directives in /etc/php/8.x/fpm/pool.d

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 79.1 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Define raw documentation and chunk it
raw_docs = [
    (
        "Nginx 502 Bad Gateway Error Resolution:\n"
        "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). "
        "Check if the php-fpm service is running using `systemctl status php-fpm`. "
        "Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
    )
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = []
for doc in raw_docs:
    chunks.extend(text_splitter.split_text(doc))

# 2. Embed chunks on CPU
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
chunk_embeddings = embed_model.encode(chunks, show_progress_bar=True, convert_to_numpy=True)

# 3. Build FAISS IndexFlatL2
dimension = chunk_embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(chunk_embeddings).astype("float32"))

# 4. Save index to disk
faiss.write_index(index, "resolution_docs.index")
print(f"Successfully indexed {index.ntotal} chunks and saved to resolution_docs.index")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Successfully indexed 1 chunks and saved to resolution_docs.index


In [ ]:
import torch
import faiss
import numpy as np
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer

# 1. Load fine-tuned adapter on GPU
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "cuda",
)
FastLanguageModel.for_inference(model)

# 2. Retrieve Context via FAISS Index File
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
index = faiss.read_index("resolution_docs.index")

user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_vector = embed_model.encode([user_query], convert_to_numpy=True).astype("float32")

distances, indices = index.search(query_vector, k=1)
retrieved_context = chunks[indices[0][0]]

# 3. Format Prompt using Official ChatML Template
messages = [
    {
        "role": "system",
        "content": (
            "You are a strict RAG extraction agent. You must answer the user query using ONLY the provided documentation context.\n"
            "Extract exact commands and file paths directly from the text. Do NOT rely on memory or output generic server configs.\n\n"
            "Required Output Structure:\n"
            "## Diagnostic Steps\n"
            "## Resolution Steps"
        )
    },
    {
        "role": "user",
        "content": f"Documentation Context:\n{retrieved_context}\n\nUser Question: {user_query}"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 4. Execute Greedy Search
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== RETRIEVED VIA FAISS ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NameError: name 'chunks' is not defined

In [ ]:
import torch
import faiss
import numpy as np
from unsloth import FastLanguageModel
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. Define raw documentation and regenerate chunks in scope
raw_docs = [
    (
        "Nginx 502 Bad Gateway Error Resolution:\n"
        "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). "
        "Check if the php-fpm service is running using `systemctl status php-fpm`. "
        "Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`."
    )
]

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1200, chunk_overlap=200)
chunks = []
for doc in raw_docs:
    chunks.extend(text_splitter.split_text(doc))

# 2. Load fine-tuned adapter on GPU
model_path = "/content/drive/MyDrive/qwen2.5-7b-it-agent"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_path,
    max_seq_length = 2048,
    load_in_4bit = True,
    device_map = "cuda",
)
FastLanguageModel.for_inference(model)

# 3. Retrieve Context via FAISS Index File
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
index = faiss.read_index("resolution_docs.index")

user_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
query_vector = embed_model.encode([user_query], convert_to_numpy=True).astype("float32")

distances, indices = index.search(query_vector, k=1)
retrieved_context = chunks[indices[0][0]]

# 4. Format Prompt using Official ChatML Template
messages = [
    {
        "role": "system",
        "content": (
            "You are a strict RAG extraction agent. You must answer the user query using ONLY the provided documentation context.\n"
            "Extract exact commands and file paths directly from the text. Do NOT rely on memory or output generic server configs.\n\n"
            "Required Output Structure:\n"
            "## Diagnostic Steps\n"
            "## Resolution Steps"
        )
    },
    {
        "role": "user",
        "content": f"Documentation Context:\n{retrieved_context}\n\nUser Question: {user_query}"
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

# 5. Execute Greedy Search
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False,
        repetition_penalty=1.2,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("=== RETRIEVED VIA FAISS ===")
print(retrieved_context)
print("\n" + "="*25 + " AGENT RESPONSE " + "="*25)
print(response)

==((====))==  Unsloth 2026.9.3: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== RETRIEVED VIA FAISS ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.

========================= AGENT RESPONSE =========================
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js).
2. Check if the php-fpm service is running using systemctl status php-fpm .
3. Verify that the unix socket path in /etc/nginx/sites-available/default matches the listen directives in /etc/php/8.x/fpm/pool

In [ ]:
import numpy as np

# ==========================================
# STEP 4.3: Standalone Retrieval Function
# ==========================================
def retrieve(query, k=1):
    """
    Embeds a user query, searches the loaded FAISS index,
    and returns top-k matching document chunks.
    """
    # 1. Encode query to float32 numpy array
    query_vector = embed_model.encode([query], convert_to_numpy=True).astype("float32")

    # 2. Search FAISS index for top-k closest vectors
    distances, indices = index.search(query_vector, k)

    # 3. Map indices back to original text chunks
    retrieved_chunks = [chunks[i] for i in indices[0] if i < len(chunks)]
    return "\n\n".join(retrieved_chunks)

# Example Verification Usage:
test_query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
retrieved_context = retrieve(test_query, k=1)

print("=== RETRIEVED CONTEXT VIA FUNCTION ===")
print(retrieved_context)

=== RETRIEVED CONTEXT VIA FUNCTION ===
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js). Check if the php-fpm service is running using `systemctl status php-fpm`. Verify that the unix socket path in `/etc/nginx/sites-available/default` matches the `listen` directives in `/etc/php/8.x/fpm/pool.d/www.conf`.


In [ ]:
# ==========================================
# PHASE 5: End-to-End Grounded Agent Pipeline
# ==========================================

def run_support_agent(user_query):
    # 1. Retrieve grounded context using FAISS
    retrieved_context = retrieve(user_query, k=1)

    # 2. Format ChatML prompt
    messages = [
        {
            "role": "system",
            "content": (
                "You are a strict RAG extraction agent. You must answer the user query using ONLY the provided documentation context.\n"
                "Extract exact commands and file paths directly from the text. Do NOT rely on memory or output generic server configs.\n\n"
                "Required Output Structure:\n"
                "## Diagnostic Steps\n"
                "## Resolution Steps"
            )
        },
        {
            "role": "user",
            "content": f"Documentation Context:\n{retrieved_context}\n\nUser Question: {user_query}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Generate response with strict decoding
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response

# Test full end-to-end execution
query = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
print(run_support_agent(query))

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server (e.g., PHP-FPM, Gunicorn, Node.js).
2. Check if the php-fpm service is running using systemctl status php-fpm .
3. Verify that the unix socket path in /etc/nginx/sites-available/default matches the listen directives in /etc/php/8.x/fpm/pool.d/www.conf .


In [ ]:
# ==========================================
# STEP 5.2: ChatML Combined Prompt Generator
# ==========================================

def generate_documentation(error_description, k=3):
    # 1. Fetch top-k contexts from FAISS
    retrieved_chunks = retrieve(error_description, k=k)
    context = "\n\n".join(retrieved_chunks) if isinstance(retrieved_chunks, list) else retrieved_chunks

    # 2. Structure prompt using Qwen ChatML system/user roles
    messages = [
        {
            "role": "system",
            "content": (
                "You are an IT support documentation assistant. Answer the query strictly using the provided reference context.\n"
                "Do NOT introduce external server paths or commands not present in the reference.\n\n"
                "Required Output Structure:\n"
                "## Diagnostic Steps\n"
                "## Resolution Steps"
            )
        },
        {
            "role": "user",
            "content": f"Reference context:\n{context}\n\nError description: {error_description}"
        }
    ]

    # 3. Apply ChatML template formatting
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # 4. Tokenize and generate
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False,
            repetition_penalty=1.2,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Test Step 5.2 execution
test_error = "Nginx is returning 502 Bad Gateway after restarting PHP-FPM."
print(generate_documentation(test_error, k=1))

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.

## Resolution Steps
1. The problem was caused by a misconfiguration of nginx's proxy_pass directive which did not match the location block for the proxied app. This resulted in requests being sent to the wrong upstream host causing the bad gateway errors.
2. To fix this I added another location block inside my existing one like so...
3. location / {
4. try_files $uri @proxy;
5. }
6. location ~ \.php$ { # Matches all .php files except those matching ^/(admin|api)/.*
7. fastcgi_split_path_info ^(.+\.php)(/.*)$; # Splits up request URI into script name and arguments passed via PATH_INFO variable
8. include snippets/fastcgi-php.conf; # Includes settings from file located at /usr/local/etc/nginx/snippets/fastcgi-php.conf
9. fastcgi_param SCRIPT_FILENAME $document_root/$fastcgi_script_name; # Sets full path to requested PHP file as value of SCRI

In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")

# Update your generation call inside generate_documentation():
output = model.generate(
    **inputs,
    max_new_tokens=768,          # Increased token cap to prevent cut-off code blocks
    generation_config=None,      # Ignores default max_length in model config
    do_sample=False,
    repetition_penalty=1.15,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.pad_token_id
)

Both `max_new_tokens` (=768) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*max_new_tokens.*")

# 1. Expand Knowledge Base with diverse IT documentation
expanded_docs = [
    "Nginx 502 Bad Gateway Error Resolution:\n"
    "A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server. "
    "Check if php-fpm is running via `systemctl status php-fpm`. "
    "Verify the unix socket path in `/etc/nginx/sites-available/default` matches `/etc/php/8.x/fpm/pool.d/www.conf`.",

    "PostgreSQL Connection Refused Error:\n"
    "Error: 'Could not connect to server: Connection refused'. "
    "Check if PostgreSQL service is active: `sudo systemctl status postgresql`. "
    "Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and "
    "`/etc/postgresql/14/main/pg_hba.conf` allows host connections.",

    "SSH Host Key Verification Failed:\n"
    "Occurs when remote server host keys change. "
    "Remove old entry using `ssh-keygen -R hostname_or_ip`. "
    "Verify fingerprint and re-establish connection via `ssh user@hostname_or_ip`.",

    "Docker Container Exited Code 137:\n"
    "Exit code 137 indicates Out of Memory (OOM) killer terminated the container. "
    "Inspect host logs using `dmesg -T | grep -i oom`. "
    "Increase memory limit in `docker-compose.yml` under `deploy.resources.limits.memory`."
]

# Chunk and rebuild index
chunks = []
for doc in expanded_docs:
    chunks.extend(text_splitter.split_text(doc))

chunk_embeddings = embed_model.encode(chunks, show_progress_bar=False, convert_to_numpy=True)
index = faiss.IndexFlatL2(chunk_embeddings.shape[1])
index.add(np.array(chunk_embeddings).astype("float32"))
faiss.write_index(index, "resolution_docs.index")

# 2. Test Cases for Manual Verification
test_queries = [
    "PostgreSQL is throwing connection refused on port 5432.",
    "Docker container keeps crashing with exit code 137.",
    "Getting SSH host key verification failed when connecting to remote server.",
    "Nginx giving 502 bad gateway after starting service."
]

# 3. Batch Evaluation Execution
for idx, q in enumerate(test_queries, 1):
    retrieved = retrieve(q, k=1)
    response = generate_documentation(q, k=1)

    print(f"==================== TEST CASE {idx} ====================")
    print(f"QUERY: {q}\n")
    print(f"--- RETRIEVED CHUNK ---\n{retrieved}\n")
    print(f"--- GENERATED OUTPUT ---\n{response}\n")

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 1 ====================
QUERY: PostgreSQL is throwing connection refused on port 5432.

--- RETRIEVED CHUNK ---
PostgreSQL Connection Refused Error:
Error: 'Could not connect to server: Connection refused'. Check if PostgreSQL service is active: `sudo systemctl status postgresql`. Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and `/etc/postgresql/14/main/pg_hba.conf` allows host connections.

--- GENERATED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.
3. Confirm the system hardware and software specifications.
4. Confirm whether any recent changes or updates occurred before this issue began.
## Resolution Steps
1. IBM PostgresSQL - Could not connect to server: Connection refused [TROUBLESHOOTING]
2. PROBLEM(ABSTRACT)
3. This document provides information about troubleshooting a "could not connect" message when try

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 2 ====================
QUERY: Docker container keeps crashing with exit code 137.

--- RETRIEVED CHUNK ---
Docker Container Exited Code 137:
Exit code 137 indicates Out of Memory (OOM) killer terminated the container. Inspect host logs using `dmesg -T | grep -i oom`. Increase memory limit in `docker-compose.yml` under `deploy.resources.limits.memory`.

--- GENERATED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.
3. Confirm the system hardware and software specifications.
4. Confirm if there is any pattern to reproduce the reported behavior.
5. Review the issue logs or error codes specified in the question.
6. Confirm the system environment and configuration details.
7. Confirm the system hardware and software specifications.
8. Confirm if there is any pattern to reproduce the reported behavior.
9. Review the issue logs or error codes specified i

Both `max_new_tokens` (=512) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 3 ====================
QUERY: Getting SSH host key verification failed when connecting to remote server.

--- RETRIEVED CHUNK ---
SSH Host Key Verification Failed:
Occurs when remote server host keys change. Remove old entry using `ssh-keygen -R hostname_or_ip`. Verify fingerprint and re-establish connection via `ssh user@hostname_or_ip`.

--- GENERATED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm the system environment and configuration details.
3. Review the issue Symptom, Likely Cause, Diagnostic Steps, Resolution Steps, Verification, and any other relevant information.
4. Confirm that you have access to the system where this problem occurred (or similar systems).
5. Confirm whether there is a maintenance window available for resolving this issue.
6. If possible, review the issue logs or error codes specified in the question.
7. Confirm the system environment and configuration details.
8. 

In [ ]:
import warnings
import torch

# Suppress HuggingFace generation warning globally
warnings.filterwarnings("ignore", category=UserWarning)

def generate_documentation_strict(error_description, k=1):
    # 1. Retrieve context
    retrieved_chunks = retrieve(error_description, k=k)
    context = "\n\n".join(retrieved_chunks) if isinstance(retrieved_chunks, list) else retrieved_chunks

    # 2. Rigid System Prompt enforcing direct extraction
    messages = [
        {
            "role": "system",
            "content": (
                "You are an automated extraction script. Your task is to output resolution steps derived STRICTLY from the provided CONTEXT.\n"
                "CRITICAL RULES:\n"
                "- Do NOT use outside knowledge.\n"
                "- Do NOT invent paths, commands, or services not written in the CONTEXT.\n"
                "- If no diagnostic steps exist in CONTEXT, write 'None provided.' under Diagnostic Steps.\n\n"
                "Output Format:\n"
                "## Diagnostic Steps\n"
                "[Steps from context or 'None provided.']\n\n"
                "## Resolution Steps\n"
                "[Exact steps from context]"
            )
        },
        {
            "role": "user",
            "content": f"CONTEXT:\n{context}\n\nQUERY: {error_description}"
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Enhanced generation parameters to stop loops and hallucinations
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.3,      # Increased to stop repeated diagnostic loops
            no_repeat_ngram_size=4,       # Enforces unique output phrases
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

# Re-test Batch Run
for idx, q in enumerate(test_queries, 1):
    retrieved = retrieve(q, k=1)
    response = generate_documentation_strict(q, k=1)

    print(f"==================== TEST CASE {idx} ====================")
    print(f"QUERY: {q}\n")
    print(f"--- RETRIEVED CHUNK ---\n{retrieved}\n")
    print(f"--- STRICT GENERATED OUTPUT ---\n{response}\n")

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 1 ====================
QUERY: PostgreSQL is throwing connection refused on port 5432.

--- RETRIEVED CHUNK ---
PostgreSQL Connection Refused Error:
Error: 'Could not connect to server: Connection refused'. Check if PostgreSQL service is active: `sudo systemctl status postgresql`. Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and `/etc/postgresql/14/main/pg_hba.conf` allows host connections.

--- STRICT GENERATED OUTPUT ---
## Diagnostic Steps

Check your firewall settings for any rules that may be blocking traffic between hosts.

Verify you have a valid route back out of the network where this error occurs (if it's internal).

If there isn't one then add:

ip rule delete fwmark >=0x8000; ip rule insert table local before all priority -999;

to /etc/rc.local so they're added at boot time as well.


## Resolution Steps

The problem was caused by SELinux policy preventing access through iptables chains. The fix I used involved

Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 2 ====================
QUERY: Docker container keeps crashing with exit code 137.

--- RETRIEVED CHUNK ---
Docker Container Exited Code 137:
Exit code 137 indicates Out of Memory (OOM) killer terminated the container. Inspect host logs using `dmesg -T | grep -i oom`. Increase memory limit in `docker-compose.yml` under `deploy.resources.limits.memory`.

--- STRICT GENERATED OUTPUT ---
## Diagnostic Steps

## Resolution Steps



Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 3 ====================
QUERY: Getting SSH host key verification failed when connecting to remote server.

--- RETRIEVED CHUNK ---
SSH Host Key Verification Failed:
Occurs when remote server host keys change. Remove old entry using `ssh-keygen -R hostname_or_ip`. Verify fingerprint and re-establish connection via `ssh user@hostname_or_ip`.

--- STRICT GENERATED OUTPUT ---
## Diagnostic Steps None provided.

## Resolution Steps ssh-hostkeychange.sh; /etc/hosts.equiv; ~/.rhost; .shosts; ~root/.netrc; $HOME/.netrc

==================== TEST CASE 4 ====================
QUERY: Nginx giving 502 bad gateway after starting service.

--- RETRIEVED CHUNK ---
Nginx 502 Bad Gateway Error Resolution:
A 502 Bad Gateway usually occurs when Nginx cannot communicate with the backend application server. Check if php-fpm is running via `systemctl status php-fpm`. Verify the unix socket path in `/etc/nginx/sites-available/default` matches `/etc/php/8.x/fpm/pool.d/www.conf`.



In [ ]:
import warnings
import torch

warnings.filterwarnings("ignore")

def generate_documentation_grounded(error_description, k=1):
    # 1. Retrieve chunk
    retrieved_chunks = retrieve(error_description, k=k)
    context = "\n\n".join(retrieved_chunks) if isinstance(retrieved_chunks, list) else retrieved_chunks

    # 2. Strict Grounding Prompting
    prompt = (
        f"<|im_start|>system\n"
        f"You are a verbatim text extractor. Extract diagnostic commands and resolution steps EXACTLY as written in the provided CONTEXT. "
        f"Do not add extra server paths, files, or steps from your training memory. If no diagnostic steps exist in CONTEXT, write 'None provided.'\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION: {error_description}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"Based strictly on the provided context:\n\n"
        f"## Diagnostic Steps\n"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Suppress max_length warnings and force strict extraction
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=200,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            # Clears default model config parameters that trigger the warning
            generation_config=None
        )

    # Decode only generated response
    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    return f"## Diagnostic Steps\n{raw_output}"

# Re-run test suite
for idx, q in enumerate(test_queries, 1):
    retrieved = retrieve(q, k=1)
    response = generate_documentation_grounded(q, k=1)

    print(f"==================== TEST CASE {idx} ====================")
    print(f"QUERY: {q}\n")
    print(f"--- RETRIEVED CHUNK ---\n{retrieved}\n")
    print(f"--- GROUNDED OUTPUT ---\n{response}\n")

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 1 ====================
QUERY: PostgreSQL is throwing connection refused on port 5432.

--- RETRIEVED CHUNK ---
PostgreSQL Connection Refused Error:
Error: 'Could not connect to server: Connection refused'. Check if PostgreSQL service is active: `sudo systemctl status postgresql`. Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and `/etc/postgresql/14/main/pg_hba.conf` allows host connections.

--- GROUNDED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.
2. Confirm that you have installed PostgresSQL correctly by running "psql" command with -d option (e.g., psql -d mydb).
3. Verify that the database name matches what's defined in the application code.
4. Confirm that the user account used for connecting to DB also exists within the database itself using \du SQL command

## Resolution Steps
1. I had this same problem when trying to run an app locally after installing it via Dock

Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 2 ====================
QUERY: Docker container keeps crashing with exit code 137.

--- RETRIEVED CHUNK ---
Docker Container Exited Code 137:
Exit code 137 indicates Out of Memory (OOM) killer terminated the container. Inspect host logs using `dmesg -T | grep -i oom`. Increase memory limit in `docker-compose.yml` under `deploy.resources.limits.memory`.

--- GROUNDED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.

## Resolution Steps
1. Exit code 137 indicates Out of Memory (OOM) killer terminated the container. Inspect host logs using dmesg -T | grep -i oom. Increase memory limit in docker-compose.yml under deploy.resources.limits.memory

## Verification
Verify that the resolution steps eliminated the reported symptom and normal operations are restored.



Both `max_new_tokens` (=200) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


==================== TEST CASE 3 ====================
QUERY: Getting SSH host key verification failed when connecting to remote server.

--- RETRIEVED CHUNK ---
SSH Host Key Verification Failed:
Occurs when remote server host keys change. Remove old entry using `ssh-keygen -R hostname_or_ip`. Verify fingerprint and re-establish connection via `ssh user@hostname_or_ip`.

--- GROUNDED OUTPUT ---
## Diagnostic Steps
1. Review the issue logs or error codes specified in the question.

## Resolution Steps
1. SSH Host Key Verification Failed:
2. Occurs when remote server host keys change. Remove old entry using ssh-keygen -R hostname_or_ip. Verify fingerprint and re-establish connection via ssh user@hostname_or_ip.

## Verification
Verify that the resolution steps eliminated the reported symptom and normal operations are restored.

==================== TEST CASE 4 ====================
QUERY: Nginx giving 502 bad gateway after starting service.

--- RETRIEVED CHUNK ---
Nginx 502 Bad Gateway Er

In [ ]:
import warnings
import torch

warnings.filterwarnings("ignore")

# Remove generation length conflict at the model level
model.generation_config.max_length = None

def generate_documentation_anchored(error_description, k=1):
    # 1. Retrieve chunk
    retrieved_chunks = retrieve(error_description, k=k)
    context = "\n\n".join(retrieved_chunks) if isinstance(retrieved_chunks, list) else retrieved_chunks

    # 2. Hard-anchored ChatML prompt
    prompt = (
        f"<|im_start|>system\n"
        f"You are a strict text extraction engine. Copy the resolution steps from the CONTEXT verbatim. "
        f"Do NOT invent steps, docker commands, or generic advice.\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n"
        f"CONTEXT:\n{context}\n\n"
        f"QUERY: {error_description}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"## Diagnostic Steps\n"
        f"1. Refer to retrieved context details below.\n\n"
        f"## Resolution Steps\n"
        f"1. "
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Deterministic greedy generation
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=150,
            do_sample=False,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode only generated response
    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

    return f"## Diagnostic Steps\n1. Refer to retrieved context details below.\n\n## Resolution Steps\n1. {raw_output}"

# Re-run test suite
for idx, q in enumerate(test_queries, 1):
    retrieved = retrieve(q, k=1)
    response = generate_documentation_anchored(q, k=1)

    print(f"==================== TEST CASE {idx} ====================")
    print(f"QUERY: {q}\n")
    print(f"--- RETRIEVED CHUNK ---\n{retrieved}\n")
    print(f"--- ANCHORED OUTPUT ---\n{response}\n")

==================== TEST CASE 1 ====================
QUERY: PostgreSQL is throwing connection refused on port 5432.

--- RETRIEVED CHUNK ---
PostgreSQL Connection Refused Error:
Error: 'Could not connect to server: Connection refused'. Check if PostgreSQL service is active: `sudo systemctl status postgresql`. Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and `/etc/postgresql/14/main/pg_hba.conf` allows host connections.

--- ANCHORED OUTPUT ---
## Diagnostic Steps
1. Refer to retrieved context details below.

## Resolution Steps
1. 1) I would check that your firewall isn't blocking it (iptables)
2. 2) If you're using SELinux then try disabling it for now until we get this working properly

## Verification
Verify that the resolution steps eliminated the reported symptom and normal operations are restored.

==================== TEST CASE 2 ====================
QUERY: Docker container keeps crashing with exit code 137.

--- RETRIEVED CHUNK ---
Docker Conta

In [ ]:
import warnings
import torch

warnings.filterwarnings("ignore")

def generate_documentation_verbatim(error_description, k=1):
    # 1. Retrieve chunk
    retrieved_chunks = retrieve(error_description, k=k)
    context = "\n\n".join(retrieved_chunks) if isinstance(retrieved_chunks, list) else retrieved_chunks

    # 2. Simplified, zero-drift extraction prompt
    prompt = (
        f"<|im_start|>system\n"
        f"Extract and list the exact commands and diagnostic steps from the context below. "
        f"Do NOT write explanatory sentences, web links, or outside knowledge. Copy directly from the text.\n"
        f"<|im_end|>\n"
        f"<|im_start|>user\n"
        f"Context:\n{context}\n\n"
        f"Extract resolution steps for: {error_description}\n"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
        f"## Diagnostic Steps\n"
        f"1. Review service logs and system status.\n\n"
        f"## Resolution Steps\n"
    )

    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")

    # 3. Controlled extraction generation
    with torch.no_grad():
        outputs = model.generate(
            inputs.input_ids,
            attention_mask=inputs.attention_mask,
            max_new_tokens=128,
            do_sample=False,
            temperature=0.0,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    raw_output = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return f"## Diagnostic Steps\n1. Review service logs and system status.\n\n## Resolution Steps\n{raw_output.strip()}"

# Re-run batch test suite
for idx, q in enumerate(test_queries, 1):
    retrieved = retrieve(q, k=1)
    response = generate_documentation_verbatim(q, k=1)

    print(f"==================== TEST CASE {idx} ====================")
    print(f"QUERY: {q}\n")
    print(f"--- RETRIEVED CHUNK ---\n{retrieved}\n")
    print(f"--- VERBATIM EXTRACTED OUTPUT ---\n{response}\n")

==================== TEST CASE 1 ====================
QUERY: PostgreSQL is throwing connection refused on port 5432.

--- RETRIEVED CHUNK ---
PostgreSQL Connection Refused Error:
Error: 'Could not connect to server: Connection refused'. Check if PostgreSQL service is active: `sudo systemctl status postgresql`. Ensure `/etc/postgresql/14/main/postgresql.conf` has `listen_addresses = '*'` and `/etc/postgresql/14/main/pg_hba.conf` allows host connections.

--- VERBATIM EXTRACTED OUTPUT ---
## Diagnostic Steps
1. Review service logs and system status.

## Resolution Steps
2. The error message you're seeing indicates that your client machine cannot reach a listening socket at all; it's possible there are other issues with the database itself (such as an invalid password) but I would start by checking whether the Postgres daemon is running properly first:

# sudo systemctl status postgress

If this shows no output then check /var/log/syslog for errors related to starting up the process - per

In [ ]:
# Test pure base model instruction following vs. LoRA adapter
print("=== WITH LORA ADAPTER ===")
print(generate_documentation_verbatim("PostgreSQL is throwing connection refused on port 5432.", k=1))

print("\n=== WITH ADAPTER DISABLED ===")
with model.disable_adapter():
    print(generate_documentation_verbatim("PostgreSQL is throwing connection refused on port 5432.", k=1))

=== WITH LORA ADAPTER ===
## Diagnostic Steps
1. Review service logs and system status.

## Resolution Steps
2. The error message you're seeing indicates that your client machine cannot reach a listening socket at all; it's possible there are other issues with the database itself (such as an invalid password) but I would start by checking whether the Postgres daemon is running properly first:

# sudo systemctl status postgress

If this shows no output then check /var/log/syslog for errors related to starting up the process - perhaps something in the configuration file isn't right? If so try restarting the service again after fixing any problems found here.

Alternatively run netstat -anp | grep :5402 which should show some information about what

=== WITH ADAPTER DISABLED ===
## Diagnostic Steps
1. Review service logs and system status.

## Resolution Steps
- Run `sudo systenctl status postgresqle`.
- Verify `listen_addressess = '*'`.
- Confirm host access in `/etc/postrgresql/1main/pg

In [1]:
# 1. Set Git credentials inside Colab
!git config --global user.name "MBee059"
!git config --global user.email "mdisodia2059@gmail.com"

# 2. Clone the repository into Colab using your PAT
# Replace YOUR_PAT with your actual token string
!git clone https://YOUR_PAT@github.com/MBee059/it-agent-.git /content/it-agent-repo

# 3. Copy your current Colab notebook into the repo folder
!cp /content/*.ipynb /content/it-agent-repo/notebooks/ 2>/dev/null || true

# 4. Commit and Push back to GitHub
%cd /content/it-agent-repo
!git add .
!git commit -m "feat: add inference and evaluation notebook from Colab"
!git push origin main

Cloning into '/content/it-agent-repo'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 2), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 5.08 KiB | 5.08 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/it-agent-repo
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://YOUR_PAT@github.com': No such device or address


In [2]:
import os
from google.colab import userdata

# 1. Safely retrieve token from Colab Secrets
github_token = userdata.get('GITHUB_TOKEN')

# 2. Configure Git identity
!git config --global user.name "MBee059"
!git config --global user.email "mdisodia2059@gmail.com"

# 3. Clone repository using the secret token
repo_url = f"https://{github_token}@github.com/MBee059/it-agent-.git"
!git clone {repo_url} /content/it-agent-repo

# 4. Create notebooks folder inside the repo if it doesn't exist
!mkdir -p /content/it-agent-repo/notebooks

# 5. Copy current notebook files into the repository folder
!cp /content/*.ipynb /content/it-agent-repo/notebooks/ 2>/dev/null || true

# 6. Commit and Push back to GitHub
%cd /content/it-agent-repo
!git add .
!git commit -m "feat: sync colab training and inference notebook"
!git push origin main

fatal: destination path '/content/it-agent-repo' already exists and is not an empty directory.
/content/it-agent-repo
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
fatal: could not read Password for 'https://YOUR_PAT@github.com': No such device or address


In [3]:
import os
import shutil
from google.colab import userdata

# 1. Retrieve the token safely from Colab Secrets
github_token = userdata.get('GITHUB_TOKEN')

# 2. Clear previous workspace to avoid destination path errors
if os.path.exists('/content/it-agent-repo'):
    shutil.rmtree('/content/it-agent-repo')

# 3. Configure Git credentials
!git config --global user.name "MBee059"
!git config --global user.email "mdisodia2059@gmail.com"

# 4. Clone repo with dynamic token insertion
repo_url = f"https://{github_token}@github.com/MBee059/it-agent-.git"
!git clone {repo_url} /content/it-agent-repo

# 5. Create notebooks directory and copy active files
!mkdir -p /content/it-agent-repo/notebooks
!cp /content/*.ipynb /content/it-agent-repo/notebooks/ 2>/dev/null || true

# 6. Commit and Push back to GitHub
%cd /content/it-agent-repo
!git add .
!git commit -m "feat: sync colab training and inference notebook"
!git push origin main

shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
Cloning into '/content/it-agent-repo'...
fatal: Unable to read current working directory: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
/content/it-agent-repo
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories)

In [4]:
import os
import shutil
from google.colab import userdata

# 1. Reset working directory to /content to fix broken working directory state
%cd /content

# 2. Safely retrieve token from Colab Secrets
github_token = userdata.get('GITHUB_TOKEN')

# 3. Clean up any existing repository folder
if os.path.exists('/content/it-agent-repo'):
    shutil.rmtree('/content/it-agent-repo')

# 4. Configure Git identity
!git config --global user.name "MBee059"
!git config --global user.email "mdisodia2059@gmail.com"

# 5. Clone repository using your secret token
repo_url = f"https://{github_token}@github.com/MBee059/it-agent-.git"
!git clone {repo_url} /content/it-agent-repo

# 6. Ensure notebooks directory exists and copy active notebooks
!mkdir -p /content/it-agent-repo/notebooks
!cp /content/*.ipynb /content/it-agent-repo/notebooks/ 2>/dev/null || true

# 7. Move into the repository directory, stage, commit, and push
%cd /content/it-agent-repo
!git add .
!git commit -m "feat: sync colab training and inference notebook"
!git push origin main

/content
Cloning into '/content/it-agent-repo'...
remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 17 (delta 2), reused 17 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (17/17), 5.08 KiB | 5.08 MiB/s, done.
Resolving deltas: 100% (2/2), done.
/content/it-agent-repo
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
